# Numerical reproduction of Tables 1--5

This notebook contains the numerical experiments reported in Tables 1--5 of [Price and Assortment Optimization under the Multinomial Logit Model with Opaque Products](https://arxiv.org/abs/2502.08124).

For every specification, 2,000 instances are generated with product counts $n=2,\ldots,9$. Valuations follow $\operatorname{Lognormal}(0,0.3)$, and prices follow the distribution stated above each table. The random seed is 48.

The first six output columns are the columns reported in the paper. The remaining columns are diagnostic statistics retained from the original numerical notebook.


In [56]:
# Imports used by the numerical experiment.
import numpy as np
import pandas as pd
from math import comb
from itertools import combinations
from functools import lru_cache
from scipy.optimize import minimize_scalar
import time

## Revenue functions

`trad_rev` evaluates traditional MNL revenue. `opq_rev` evaluates revenue with an opaque product using the inclusion--exclusion representation for the $k$th order statistic.


In [57]:
# Core revenue functions for a fixed assortment and opaque price.
def trad_rev(r, v):
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    if r.size == 0:
        return 0.0
    w = np.exp(v - r)
    return float(np.dot(r, w) / (1.0 + w.sum()))


# helper: inclusion-exclusion terms for exact opaque revenue
@lru_cache(None)
def _ie_terms(m, k):
    terms = []
    idx = range(m)
    for s in range(k, m + 1):
        coef = (-1) ** (s - k) * comb(s - 1, k - 1)
        for I in combinations(idx, s):
            terms.append((I, coef))
    return terms


# ---------- (2) opaque revenue for rho <= r_(k)^S ----------
def opq_rev(r, v, rho, k):
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    m = r.size
    if m == 0:
        return 0.0

    k_eff = min(k, m)
    ub = np.partition(r, k_eff - 1)[k_eff - 1]
    if rho < 0 or rho > ub + 1e-12:
        raise ValueError("This function only handles rho in [0, r_(k)^S].")

    total = 0.0
    for I, coef in _ie_terms(m, k_eff):
        rr = r.copy()
        I = list(I)
        rr[I] = np.minimum(rr[I], rho)
        total += coef * trad_rev(rr, v)
    return float(total)


# ---------- (3) best opaque price in [0, r_(k)^S] ----------
# def opt_opaque_price(r, v, k, xatol=1e-6, maxiter=1000):
#     r = np.asarray(r, dtype=float)
#     v = np.asarray(v, dtype=float)
#     m = r.size
#     if m == 0:
#         return 0.0, 0.0

#     k_eff = min(k, m)
#     ub = np.partition(r, k_eff - 1)[k_eff - 1]
#     if ub <= 0:
#         return 0.0, opq_rev(r, v, 0.0, k_eff)

#     f = lambda x: -opq_rev(r, v, x, k_eff)
#     res = minimize_scalar(
#         f,
#         bounds=(0.0, float(ub)),
#         method="bounded",
#         options={"xatol": xatol, "maxiter": maxiter},
#     )

#     # also check endpoints explicitly
#     cand_rhos = [0.0, float(ub), float(res.x)]
#     cand_vals = [opq_rev(r, v, x, k_eff) for x in cand_rhos]
#     j = int(np.argmax(cand_vals))
#     return cand_rhos[j], float(cand_vals[j])


## Optimizing the opaque price

The next two cells retain the two optimizer implementations from the original notebook. The first searches every interval between price breakpoints. The second redefines `opt_opaque_price` and is therefore the active implementation used by Tables 1--5; it checks every breakpoint and searches the intervals adjacent to the best breakpoint.


In [ ]:
# Reference implementation: search every interval between consecutive breakpoints.
def opt_opaque_price(r, v, k, xatol=1e-6, maxiter=1000):
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    m = r.size
    if m == 0:
        return 0.0, 0.0

    k_eff = min(k, m)
    ub = np.partition(r, k_eff - 1)[k_eff - 1]

    if ub <= 0:
        return 0.0, opq_rev(r, v, 0.0, k_eff)

    # Breakpoints consist of 0, all assortment prices not exceeding the upper bound, and the upper bound.
    pts = sorted(set([0.0] + [float(x) for x in r if x <= ub] + [float(ub)]))

    # Evaluate all interval endpoints.
    vals = [opq_rev(r, v, x, k_eff) for x in pts]
    best_idx = int(np.argmax(vals))
    best_rho = float(pts[best_idx])
    best_rev = float(vals[best_idx])

    best_rho = 0.0
    best_rev = opq_rev(r, v, 0.0, k_eff)

    f = lambda x: -opq_rev(r, v, x, k_eff)

    for a, b in zip(pts[:-1], pts[1:]):
        # Check both endpoints first.
        for x in (a, b):
            val = opq_rev(r, v, x, k_eff)
            if val > best_rev:
                best_rev = val
                best_rho = x

        # Then perform a bounded search inside the current interval.
        if b - a > 1e-12:
            res = minimize_scalar(f,  bounds=(a, b),method="bounded", options={"xatol": xatol, "maxiter": maxiter}, )
            x = float(res.x)
            val = opq_rev(r, v, x, k_eff)
            if val > best_rev:
                best_rev = val
                best_rho = x

    return best_rho, float(best_rev)

In [66]:
# Active implementation: search the intervals adjacent to the best breakpoint.
def opt_opaque_price(r, v, k, xatol=1e-6, maxiter=1000):
    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    m = r.size
    if m == 0:
        return 0.0, 0.0

    k_eff = min(k, m)
    ub = np.partition(r, k_eff - 1)[k_eff - 1]

    if ub <= 0:
        return 0.0, opq_rev(r, v, 0.0, k_eff)

    pts = sorted(set([0.0] + [float(x) for x in r if x <= ub] + [float(ub)]))

    # Evaluate all breakpoints first.
    vals = [opq_rev(r, v, x, k_eff) for x in pts]
    best_idx = int(np.argmax(vals))
    best_rho = float(pts[best_idx])
    best_rev = float(vals[best_idx])

    f = lambda x: -opq_rev(r, v, x, k_eff)

    # Search only the intervals immediately adjacent to the best breakpoint.
    candidate_intervals = []
    if best_idx > 0:
        candidate_intervals.append((pts[best_idx - 1], pts[best_idx]))
    if best_idx < len(pts) - 1:
        candidate_intervals.append((pts[best_idx], pts[best_idx + 1]))

    for a, b in candidate_intervals:
        if b - a <= 1e-12:
            continue
        res = minimize_scalar(f, bounds=(a, b), method="bounded", options={"xatol": xatol, "maxiter": maxiter},)
        x = float(res.x)
        val = opq_rev(r, v, x, k_eff)
        if val > best_rev:
            best_rev = val
            best_rho = x

    return best_rho, float(best_rev)

## Assortment search

For each candidate assortment, `_eval_subset` chooses between traditional selling and opaque selling. `best_nrv_assortment` searches the NRV family, while `brute_force_opt` enumerates all candidate assortments to obtain the benchmark optimum.


In [59]:
# Evaluate one assortment, construct the NRV family, and compute the brute-force benchmark.
def _eval_subset(v, r, S, k):
    S = np.asarray(S, dtype=int)
    rs = np.asarray(r, dtype=float)[S]
    vs = np.asarray(v, dtype=float)[S]

    rev_trad = trad_rev(rs, vs)
    k_eff = min(k, len(rs))
    r_k = np.partition(rs, k_eff - 1)[k_eff - 1]
    rho_star, rev_opq = opt_opaque_price(rs, vs, k)
    uses_opaque = (rho_star < r_k - 1e-6) and (rev_opq > rev_trad + 1e-10)

    if uses_opaque:
        return {
            "S": S,
            "rev": rev_opq,
            "rho": rho_star,
            "uses_opaque": True,
        }
    else:
        return {
            "S": S,
            "rev": rev_trad,
            "rho": None,
            "uses_opaque": False,
        }

    # if rev_opq > rev_trad + 1e-8:
    #     return {
    #         "S": S,
    #         "rev": rev_opq,
    #         "rho": rho_star,
    #         "uses_opaque": True,
    #     }
    # else:
    #     return {
    #         "S": S,
    #         "rev": rev_trad,
    #         "rho": None,
    #         "uses_opaque": False,
    #     }


# ---------- (4) best NRV assortment ----------
def best_nrv_assortment(v, r, k):
    v = np.asarray(v, dtype=float)
    r = np.asarray(r, dtype=float)
    n = r.size

    order_r = np.argsort(-r)  # Sort products by decreasing price.
    order_v = np.argsort(-v)  # Sort products by decreasing valuation.

    best = {"S": np.array([], dtype=int), "rev": 0.0, "rho": None, "uses_opaque": False}
    seen = set()

    for i in range(1, n + 1):
        Sr = set(order_r[:i])
        for j in range(1, n + 1):
            Sv = set(order_v[:j])
            S = tuple(sorted(Sr & Sv))
            if not S or S in seen:
                continue
            seen.add(S)

            cand = _eval_subset(v, r, np.array(S, dtype=int), k)
            if cand["rev"] > best["rev"] + 1e-8:
                best = cand

    return best


# ---------- (5) brute-force optimal assortment ----------
def brute_force_opt(v, r, k, subset_list=None):
    v = np.asarray(v, dtype=float)
    r = np.asarray(r, dtype=float)
    n = r.size

    if subset_list is None:
        subset_list = [
            np.array(S, dtype=int)
            for m in range(1, n + 1)
            for S in combinations(range(n), m)
        ]

    best = {"S": np.array([], dtype=int), "rev": 0.0, "rho": None, "uses_opaque": False}
    for S in subset_list:
        cand = _eval_subset(v, r, S, k)
        if cand["rev"] > best["rev"] + 1e-8:
            best = cand
    return best




## Instance generation and experiment summary

A single $2000\times9$ instance matrix is generated for each specification. For a given $n$, only the first $n$ products are used. The exhaustive search is accelerated by reusing the optimum for $n-1$ and checking only assortments containing the newly added product.


In [60]:
# Generate the common instance bed and cache subsets used in exhaustive enumeration.
def get_instances( n_instances=2000, n_max=9, mu_r=0.5, sigma_r=1.5, mu_v=0.0, sigma_v=0.3, seed=None,):
    if seed is not None:
        np.random.seed(seed)

    v = np.random.lognormal(mean=mu_v, sigma=sigma_v, size=(n_instances, n_max))
    r = np.random.lognormal(mean=mu_r, sigma=sigma_r, size=(n_instances, n_max))

    while np.any(r > 300):
        r = np.random.lognormal(mean=mu_r, sigma=sigma_r, size=(n_instances, n_max))

    return v, r

# Cache assortment index sets for faster enumeration.
def _all_subsets(n):
    return [ np.array([i for i in range(n) if (mask >> i) & 1], dtype=int) for mask in range(1, 1 << n) ]


def _subsets_with_last(n):
    last_bit = 1 << (n - 1)
    return [ np.array([i for i in range(n) if (mask >> i) & 1], dtype=int)  for mask in range(1, 1 << n)  if mask & last_bit  ]

In [61]:
# Run one complete specification and return its summary and instance-level details.
def compute_results(  k, n_instances=2000, n_max=9, mu_r=0.5, sigma_r=1.5, mu_v=0.0, sigma_v=0.3, seed=0,):

    V_all, R_all = get_instances( n_instances=n_instances,  n_max=n_max, mu_r=mu_r,sigma_r=sigma_r, mu_v=mu_v,sigma_v=sigma_v, seed=seed )

    # Cache all subsets and the subsets containing the new product when moving from n-1 to n.
    all_subsets = {n: _all_subsets(n) for n in range(1, n_max + 1)}
    new_subsets = {1: all_subsets[1]}
    for n in range(2, n_max + 1):
        new_subsets[n] = _subsets_with_last(n)

    stats = {
        n: {
            "n": n,
            "n_instances_w_opaque": 0,
            "n_suboptimal": 0,
            "max_gap_pct": 0.0,
            "sum_gap_pct": 0.0,
            "sum_opt_size": 0.0,
            "sum_opaque_lift_pct": 0.0,
            "sum_trad_rev": 0.0,
            "sum_opq_rev": 0.0,
            "runtime_sec": 0.0,
        }
        for n in range(2, n_max + 1)
    }
    opt_records = []
    

    # Store the previous optimum for each instance so it can be reused at the next product count.
    prev_opts = [None] * n_instances

    for n in range(2, n_max + 1):
        tic = time.perf_counter()

        for t in range(n_instances):
            v = V_all[t, :n]
            r = R_all[t, :n]
            prev_opt = prev_opts[t]

            # The optimum for n is either the previous optimum or an assortment containing the new product.
            if prev_opt is None:
                best_new = brute_force_opt(v, r, k, subset_list=all_subsets[n])
                opt = best_new
            else:
                best_new = brute_force_opt(v, r, k, subset_list=new_subsets[n])
                opt = best_new if best_new["rev"] > prev_opt["rev"] + 1e-8 else prev_opt

            prev_opts[t] = opt
            nrv = best_nrv_assortment(v, r, k)

            gap_pct = 100.0 * max(0.0, opt["rev"] - nrv["rev"]) / opt["rev"]

            opt_records.append(
                    {
                        "instance": t,
                        "n": n,
                        "opt_S": tuple(opt["S"].tolist()),
                        "opt_size": len(opt["S"]),
                        "opt_rev": opt["rev"],
                        "opt_rho": opt["rho"],
                        "opt_uses_opaque": opt["uses_opaque"],
                        "nrv_S": tuple(nrv["S"].tolist()),
                        "nrv_size": len(nrv["S"]),
                        "nrv_rev": nrv["rev"],
                        "nrv_rho": nrv["rho"],
                        "nrv_uses_opaque": nrv["uses_opaque"],
                        "gap_pct": gap_pct,
                    }
                            )
            st = stats[n]
            rs_opt = r[opt["S"]]
            vs_opt = v[opt["S"]]
            trad_opt = trad_rev(rs_opt, vs_opt)

            st["sum_trad_rev"] += trad_opt
            st["sum_opq_rev"] += opt["rev"]
            st["n_instances_w_opaque"] += int(opt["uses_opaque"])
            st["n_suboptimal"] += int(gap_pct > 1e-9)
            st["max_gap_pct"] = max(st["max_gap_pct"], gap_pct)
            st["sum_gap_pct"] += gap_pct
            st["sum_opt_size"] += len(opt["S"])

            if opt["uses_opaque"]:
                # lift_pct = 100.0 * (opt["rev"] - trad_opt) / max(trad_opt, 1e-12)
                lift_add =  opt["rev"] - trad_opt
                st["sum_opaque_lift_pct"] += lift_add

        stats[n]["runtime_sec"] = time.perf_counter() - tic

    rows = []
    for n in range(2, n_max + 1):
        st = stats[n]
        rows.append(
            {
                "n": n,
                "# instances w/ opaque": st["n_instances_w_opaque"],
                "# sub-optimal": st["n_suboptimal"],
                "max. opt gap (%)": round(st["max_gap_pct"], 4),
                "avg opt gap (%)": round(st["sum_gap_pct"] / n_instances, 4),
                "avg size |S*|": round(st["sum_opt_size"] / n_instances, 2),
                "avg trad revenue": round(st["sum_trad_rev"] / n_instances, 2 ),
                "avg opq revenue": round(st["sum_opq_rev"] / n_instances, 2 ),
                "avg opaque lift": round( st["sum_opaque_lift_pct"] / max(st["n_instances_w_opaque"], 1), 4),
                "runtime (sec)": round(st["runtime_sec"], 4),
            }
        )
    summary_df = pd.DataFrame(rows)
    details_df = pd.DataFrame(opt_records)    

    return summary_df, details_df

## Table 1

$k=1$ and prices follow $\operatorname{Lognormal}(0.5,1.5)$.


In [6]:
summary_df1, details_df1 = compute_results(k=1, n_instances=2000, n_max=9, mu_r=0.5, sigma_r=1.5, mu_v=0.0, sigma_v=0.3, seed=48)
summary_df1

,n,# instances w/ opaque,# sub-optimal,max. opt gap (%),avg opt gap (%),avg size |S*|,avg trad revenue,avg opq revenue,avg opaque lift,runtime (sec)
0,2,1278,0,0.0000,0.0000,1.32,0.37,0.61,0.3779,3.4781
1,3,1248,15,2.4978,0.0066,1.74,0.45,0.69,0.3829,8.2905
2,4,1110,24,1.5620,0.0068,2.25,0.55,0.76,0.3826,22.2500
3,5,1018,41,2.0777,0.0086,2.74,0.62,0.81,0.3802,58.5644
4,6,918,54,2.0777,0.0094,3.28,0.70,0.86,0.3569,159.2337
5,7,826,59,1.9671,0.0124,3.85,0.77,0.91,0.3365,443.0035
6,8,728,64,1.9493,0.0098,4.43,0.84,0.96,0.3248,1221.8969
7,9,672,79,2.2990,0.0083,4.96,0.90,1.00,0.2962,3461.9822


## Table 2

$k=1$ and prices follow $\operatorname{Lognormal}(1,1.5)$.


In [7]:
summary_df1_1, details_df1_1 = compute_results(k=1, n_instances=2000, n_max=9, mu_r=1, sigma_r=1.5, mu_v=0.0, sigma_v=0.3, seed=48)
summary_df1_1

,n,# instances w/ opaque,# sub-optimal,max. opt gap (%),avg opt gap (%),avg size |S*|,avg trad revenue,avg opq revenue,avg opaque lift,runtime (sec)
0,2,1513,0,0.0000,0.0000,1.30,0.31,0.63,0.4247,3.1614
1,3,1470,17,1.7197,0.0058,1.69,0.40,0.71,0.4244,7.5488
2,4,1336,44,1.7407,0.0097,2.22,0.49,0.77,0.4168,20.4762
3,5,1235,96,2.8752,0.0184,2.79,0.58,0.82,0.3840,54.7965
4,6,1116,123,2.8752,0.0191,3.43,0.67,0.87,0.3607,151.5339
5,7,1005,133,3.5219,0.0171,4.07,0.75,0.92,0.3371,423.8333
6,8,917,148,1.8808,0.0134,4.71,0.82,0.96,0.3134,1205.7621
7,9,835,139,1.6723,0.0096,5.36,0.88,1.00,0.2961,3426.7000


## Table 3

$k=1$ and prices follow $\operatorname{Lognormal}(0.5,1)$.


In [15]:
summary_df1_2, details_df1_2 = compute_results(k=1, n_instances=2000, n_max=9, mu_r=0.5, sigma_r=1, mu_v=0.0, sigma_v=0.3, seed=48)
summary_df1_2

,n,# instances w/ opaque,# sub-optimal,max. opt gap (%),avg opt gap (%),avg size |S*|,avg trad revenue,avg opq revenue,avg opaque lift,runtime (sec)
0,2,1068,0,0.0000,0.0000,1.53,0.50,0.65,0.2874,3.5404
1,3,883,8,2.1268,0.0040,2.15,0.63,0.75,0.2866,9.2867
2,4,728,13,2.5867,0.0032,2.80,0.74,0.84,0.2735,23.2274
3,5,594,18,1.2539,0.0038,3.47,0.84,0.91,0.2415,62.2904
4,6,506,20,1.2539,0.0025,4.10,0.92,0.97,0.2124,169.1772
5,7,392,13,0.5850,0.0008,4.72,1.00,1.03,0.1855,464.0051
6,8,321,15,0.5850,0.0005,5.28,1.06,1.09,0.1649,1281.2909
7,9,300,12,0.1427,0.0003,5.80,1.11,1.13,0.1499,3657.6222


## Table 4

$k=3$ and prices follow $\operatorname{Lognormal}(0.5,1.5)$.


In [68]:
summary_df3, details_df3 = compute_results(k=3, n_instances=2000, n_max=9, mu_r=0.5, sigma_r=1.5, mu_v=0.0, sigma_v=0.3, seed=48)
summary_df3

,n,# instances w/ opaque,# sub-optimal,max. opt gap (%),avg opt gap (%),avg size |S*|,avg trad revenue,avg opq revenue,avg opaque lift,runtime (sec)
0,2,1497,0,0.0000,0.0000,1.61,0.44,0.68,0.3282,3.2562
1,3,1712,0,0.0000,0.0000,2.20,0.54,0.83,0.3366,5.9913
2,4,1836,7,2.7053,0.0041,2.60,0.61,0.94,0.3550,13.2963
3,5,1890,9,2.7053,0.0035,2.84,0.64,1.01,0.3892,34.3704
4,6,1924,18,3.4438,0.0079,3.00,0.66,1.06,0.4204,97.6725
5,7,1932,33,2.7053,0.0087,3.16,0.68,1.10,0.4370,299.2657
6,8,1916,51,2.4450,0.0094,3.34,0.70,1.13,0.4553,920.6063
7,9,1894,68,3.7969,0.0113,3.54,0.71,1.16,0.4721,2762.5171


## Table 5

$k=7$ and prices follow $\operatorname{Lognormal}(0.5,1.5)$.


In [ ]:
# Run the k=7 specification used in Table 5.
summary_df7, details_df7 = compute_results(k=7, n_instances=2000, n_max=9, mu_r=0.5, sigma_r=1.5, mu_v=0.0, sigma_v=0.3, seed=48)
summary_df7

,n,# instances w/ opaque,# sub-optimal,max. opt gap (%),avg opt gap (%),avg size |S*|,avg trad revenue,avg opq revenue,avg opaque lift,runtime (sec)
0,2,1497,0,0.000,0.0000,1.61,0.44,0.68,0.3282,4.4160
1,3,1712,0,0.000,0.0000,2.20,0.54,0.83,0.3366,10.2532
2,4,1841,0,0.000,0.0000,2.74,0.63,0.95,0.3424,24.1262
3,5,1909,0,0.000,0.0000,3.25,0.70,1.04,0.3566,51.3752
4,6,1949,0,0.000,0.0000,3.74,0.76,1.12,0.3725,108.9150
5,7,1971,0,0.000,0.0000,4.22,0.82,1.20,0.3858,231.6732
6,8,1981,0,0.000,0.0000,4.69,0.87,1.26,0.3965,525.8237
7,9,1984,3,1.265,0.0008,5.13,0.91,1.32,0.4104,1386.4881
